In [4]:
import pandas as pd
import os
import joblib
from gene_thesaurus import GeneThesaurus

In [2]:
MODELS_PATH = "/home/mariya/Data/pallas/models/gex"
model_path = os.path.join(MODELS_PATH, "final_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")

model = joblib.load(model_path)
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

In [7]:
importances = model.feature_importances_

# Rank cpg sites by importance
importances_df = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(importances_df["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
importances_df["Gene symbol"] = importances_df["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(importances_df.head(20))


Top 20 genes driving classification (final model):
                 Gene  Importance Gene symbol
4643  ENSG00000184507    0.028297       NUTM1
461   ENSG00000079102    0.026084     RUNX1T1
2637  ENSG00000145794    0.018835      MEGF10
5496  ENSG00000248329    0.018226       APELA
732   ENSG00000101115    0.017370       SALL4
1072  ENSG00000108511    0.011917       HOXB6
940   ENSG00000105642    0.011721       KCNN1
3082  ENSG00000157680    0.011458        DGKI
4672  ENSG00000185046    0.010392      ANKS1B
1484  ENSG00000120075    0.009492       HOXB5
3724  ENSG00000168004    0.009181      PLAAT5
3428  ENSG00000164100    0.008383       NDST3
1775  ENSG00000128510    0.008078        CPA4
5599  ENSG00000259803    0.007855    SLC22A31
2550  ENSG00000143995    0.007552       MEIS1
1637  ENSG00000124302    0.007545       CHST8
376   ENSG00000072657    0.007475       TRHDE
2535  ENSG00000143816    0.007365       WNT9A
847   ENSG00000103449    0.007070       SALL1
10    ENSG00000002745    0.0

In [8]:
# Filter out all importances that are zero
non_zero_importances_df = importances_df[importances_df["Importance"] > 0]
print(f"\nNumber of genes with non-zero importance: {non_zero_importances_df.shape[0]}")


Number of genes with non-zero importance: 5859


In [9]:
# Save the ranked non-zero importances to a CSV file for further analysis
output_path = os.path.join(MODELS_PATH, "gene_importances.csv")
non_zero_importances_df.to_csv(output_path, index=False)
print(f"\nRanked gene importances saved to: {output_path}")

# Also save just the features names with non-zero importance
# as a txt file of the same format as selected_features.txt
# Name it selected_features.filtered.sorted.txt
filtered_sorted_features_path = os.path.join(MODELS_PATH, "selected_features.filtered.sorted.txt")
non_zero_importances_df[non_zero_importances_df["Importance"] > 0]["Gene"].to_csv(filtered_sorted_features_path, index=False, header=False)
print(f"\nFiltered and sorted feature names saved to: {filtered_sorted_features_path}")


Ranked gene importances saved to: /home/mariya/Data/pallas/models/gex/gene_importances.csv

Filtered and sorted feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.filtered.sorted.txt


In [10]:
# Save the first 500 and 1000 features with non-zero importance to separate txt files
top_500_features_path = os.path.join(MODELS_PATH, "selected_features.top_500.txt")
top_1000_features_path = os.path.join(MODELS_PATH, "selected_features.top_1000.txt")
non_zero_importances_df.head(500)["Gene"].to_csv(top_500_features_path, index=False, header=False)
non_zero_importances_df.head(1000)["Gene"].to_csv(top_1000_features_path, index=False, header=False)
print(f"\nTop 500 feature names saved to: {top_500_features_path}")
print(f"Top 1000 feature names saved to: {top_1000_features_path}")


Top 500 feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.top_500.txt
Top 1000 feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.top_1000.txt
